# 03 Rule Classification

Apply deterministic first-pass rules to the latest inventory output and produce a review table. Still dry-run only.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_5.yaml'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH =', POLICY_PATH)


PROJECT_ROOT = c:\00_dev\SCH-FILE-ORGANIZER
POLICY_PATH = c:\00_dev\SCH-FILE-ORGANIZER\policy\SCH_fileserver_policy_v2_5.yaml


In [2]:
from datetime import datetime
import pandas as pd

from src.policy_loader import PolicyLoader
from src.rules import classify_inventory, save_rule_outputs

policy = PolicyLoader.from_file(POLICY_PATH)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


ImportError: cannot import name 'save_rule_outputs' from 'src.rules' (c:\00_dev\SCH-FILE-ORGANIZER\src\rules.py)

In [3]:
inventory_files = list(OUTPUT_DIR.glob('inventory_*.parquet'))
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'

# Pick the newest file by filesystem timestamp, not filename order.
latest_inventory = max(inventory_files, key=lambda p: p.stat().st_mtime)
print('Using inventory:', latest_inventory.name)
inv = pd.read_parquet(latest_inventory)
print('Rows:', len(inv))
print('Columns:', list(inv.columns))


NameError: name 'OUTPUT_DIR' is not defined

## Schema note
`classify_inventory()` now backfills missing inventory fields such as `filename`, `suffix`, `parent_relative`, `path_length`, and `filename_length` if you loaded an older inventory parquet. For best consistency, rerun `02_inventory.ipynb` after policy or scanner changes.


In [49]:
classified = classify_inventory(inv, policy)
classified[['relative_path', 'rule_status', 'rule_reason', 'rule_confidence', 'proposed_relative_target']].head(5)

,relative_path,rule_status,rule_reason,rule_confidence,proposed_relative_target
0,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...
1,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...
2,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...
3,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...
4,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...


In [50]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'rule_classification_{stamp}'
csv_path, parquet_path = save_rule_outputs(classified, output_base)
print('CSV:', csv_path)
print('Parquet:', parquet_path)


CSV: c:\00_Developement\sch-file-organizer\data\outputs\rule_classification_20260308_130634.csv
Parquet: c:\00_Developement\sch-file-organizer\data\outputs\rule_classification_20260308_130634.parquet


In [51]:
classified.groupby('rule_status').size().sort_values(ascending=False).to_frame('count')

,count
rule_status,
review,1676
move_to_special_folder,130


In [52]:
classified[classified['rule_status'] == 'archive_or_delete_candidate'][['relative_path', 'filename', 'rule_reason', 'proposed_relative_target']].head(10)

,relative_path,filename,rule_reason,proposed_relative_target


In [53]:
classified[classified['rule_status'] == 'move_to_special_folder'][['relative_path', 'filename', 'rule_reason', 'special_folder_target', 'proposed_relative_target']].head(10)

,relative_path,filename,rule_reason,special_folder_target,proposed_relative_target
0,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0004_NYSE_RDN_2012_p155.pdf,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
1,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0004_NYSE_RDN_2012_p155.txt,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
2,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0005_NYSE_TDW_2010_p054.pdf,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
3,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0005_NYSE_TDW_2010_p054.txt,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
4,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0006_pilot_handbook_p032.pdf,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
5,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0006_pilot_handbook_p032.txt,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
6,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0007_2021-atp-rulebook-25may_p03...,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
7,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0007_2021-atp-rulebook-25may_p03...,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
8,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0008_1002.1420_p023.pdf,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...
9,01_selected\random_1_dedup_target\doclaynet_pd...,doclaynet_pdf_0008_1002.1420_p023.txt,duplicate hash non-canonical copy,_DUPLICATED,_DUPLICATED/01_selected\random_1_dedup_target/...


In [54]:
classified[classified['rule_status'] == 'compliant_keep_review_path'][['relative_path', 'filename', 'parsed_phase', 'parsed_doc_type', 'default_folder_subpath', 'proposed_relative_target']].head(10)

,relative_path,filename,parsed_phase,parsed_doc_type,default_folder_subpath,proposed_relative_target


In [55]:
classified[classified['rule_status'] == 'review'][['relative_path', 'filename', 'rule_reason', 'path_risk', 'filename_risk']].head(10)

,relative_path,filename,rule_reason,path_risk,filename_risk
130,01_selected\New Text Document.ogb,New Text Document.ogb,needs classification or rename mapping,False,False
131,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,doclaynet_pdf_0001_refman-8.0-en_p3115.pdf,needs classification or rename mapping,False,False
132,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,doclaynet_pdf_0001_refman-8.0-en_p3115.txt,needs classification or rename mapping,False,False
133,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,doclaynet_pdf_0002_pilot_handbook_p257.pdf,needs classification or rename mapping,False,False
134,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,doclaynet_pdf_0002_pilot_handbook_p257.txt,needs classification or rename mapping,False,False
135,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,doclaynet_pdf_0003_1002.3753_p005.pdf,needs classification or rename mapping,False,False
136,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,doclaynet_pdf_0003_1002.3753_p005.txt,needs classification or rename mapping,False,False
137,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,doclaynet_pdf_0004_NYSE_RDN_2012_p155.pdf,needs classification or rename mapping,False,False
138,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,doclaynet_pdf_0004_NYSE_RDN_2012_p155.txt,needs classification or rename mapping,False,False
139,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,doclaynet_pdf_0005_NYSE_TDW_2010_p054.pdf,needs classification or rename mapping,False,False


In [57]:
classified.sort_values(['action_priority', 'relative_path']).head(10)

,scan_root,absolute_path,relative_path,parent_relative,filename,stem,suffix,size_bytes,modified_at,created_at,...,parsed_version,parsed_status,parsed_ext,default_folder_subpath,special_folder_target,rule_status,rule_reason,rule_confidence,proposed_relative_target,action_priority
0,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0004_NYSE_RDN_2012_p155.pdf,doclaynet_pdf_0004_NYSE_RDN_2012_p155,.pdf,273096,2026-03-08 04:32:19.148512602,2026-03-08 09:19:51.488616228,...,None,None,None,None,_DUPLICATED,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...,2
1,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0004_NYSE_RDN_2012_p155.txt,doclaynet_pdf_0004_NYSE_RDN_2012_p155,.txt,5646,2026-03-08 04:32:19.149513006,2026-03-08 09:19:51.495893002,...,None,None,None,None,_DUPLICATED,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...,2
2,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0005_NYSE_TDW_2010_p054.pdf,doclaynet_pdf_0005_NYSE_TDW_2010_p054,.pdf,60106,2026-03-08 04:32:19.150513887,2026-03-08 09:19:51.500800370,...,None,None,None,None,_DUPLICATED,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...,2
3,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0005_NYSE_TDW_2010_p054.txt,doclaynet_pdf_0005_NYSE_TDW_2010_p054,.txt,2232,2026-03-08 04:32:19.151512384,2026-03-08 09:19:51.507491350,...,None,None,None,None,_DUPLICATED,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...,2
4,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0006_pilot_handbook_p032.pdf,doclaynet_pdf_0006_pilot_handbook_p032,.pdf,183832,2026-03-08 04:32:19.153232574,2026-03-08 09:19:51.514171838,...,None,None,None,None,_DUPLICATED,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...,2
5,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0006_pilot_handbook_p032.txt,doclaynet_pdf_0006_pilot_handbook_p032,.txt,3197,2026-03-08 04:32:19.155240536,2026-03-08 09:19:51.520810604,...,None,None,None,None,_DUPLICATED,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...,2
6,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0007_2021-atp-rulebook-25may_p03...,doclaynet_pdf_0007_2021-atp-rulebook-25may_p035,.pdf,125583,2026-03-08 04:32:19.157250643,2026-03-08 09:19:51.527583838,...,None,None,None,None,_DUPLICATED,move_to_special_folder,duplicate hash non-canonical copy,0.98,_DUPLICATED/01_selected\random_1_dedup_target/...,2
7,C:\Users\User\Desktop\Random_Files_WORKING_COPY,C:\Users\User\Desktop\Random_Files_WORKING_COP...,01_selected\random_1_dedup_target\doclaynet_pd...,01_selected\random_1_dedup_target,doclaynet_pdf_0007_2021-atp-rulebook-25may_p03...,doclaynet_pdf_0007_2021-atp-rulebook-25may_p035,.txt,8218,2026-03-08 